# Fine-tuning ModernBERT with LoRA: Signpost

**Part 4 of "LLMs the Hard Way"** (branch `signpost`). `02_frozen_router.ipynb` routed conversations well with a **frozen** encoder, but grounded questions about a text plateaued at 114 / 170: no head or amount of data could make frozen features *read*. Here the encoder itself learns, through **LoRA** adapters on every attention and MLP layer of ModernBERT-large (7.2 M trained parameters next to 395 M frozen), plus a small scoring head.

**One model, two jobs**, in one question format, with one forward pass per option:
- **agent routing** with chat history and the current agent: "Which agent should handle the user's latest message?";
- **grounded questions** about a text: yes / no, one of N options, or a level on an ordered scale.

```text
<role>: <msg> [SEP] … [SEP] current agent: X [SEP] question: <q> [SEP] candidate: <option>. <description>
        -> ModernBERT-large (+ LoRA) -> mean pool -> head -> score;   softmax over the options -> answer
```

The notebook trains that model, tests it, and saves it as **Signpost**: the published model at [huggingface.co/skamalj/signpost](https://huggingface.co/skamalj/signpost) came from this recipe (run 25; tag `signpost-v1.0`). A rerun trains a new model of the same recipe; it is compared with run 25, and nothing is uploaded.

**Tests** (never used for training):

| Test | What it contains | Items |
|---|---|---|
| E10 | Realistic chats at 13 companies, routed live (the model's own previous pick is the current agent) | 193 turns; 83 at the 6 test companies |
| E11 | One opening message, 5 agents | 100 |
| Banking | Banking conversations, 5 banking agents | 50 · 225 |
| E12 | Grounded questions; the answer is always in the text | 170 (58 yes / no, 59 choice, 53 score) |
| Independent | Written by a separate author from a brief: questions, live routing, first messages | 300 · 172 turns · 200 |
| Independent 2 | A second outside set: questions incl. 10–50 options, 8–15 agents per company | 200 · 70 turns · 50 |

**Latency** is measured on every test, one question at a time, warm, tokenization included (p50 / p95 in ms). **Compared with** run 25 (the published Signpost), the frozen system (`02`, run 17) and the 2B [Strands decider](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19) (`04_decider_benchmark.ipynb`).

## Cell 1 — Setup

The defaults are Signpost's recipe (run 25). All settings are environment variables:

| Setting | Default | Meaning |
|---|---|---|
| `LORA_R`, `LORA_ALPHA` | 16, 32 | LoRA rank and scale |
| `LR`, `HEAD_LR` | 2e-4, 1e-3 | learning rates of the adapters and of the head |
| `EPOCHS` | 2 | passes over all training questions |
| `BATCH_ROUTE`, `BATCH_CHOICE` | 4, 16 | questions per step (a routing question brings 5 longer inputs, a choice question 2–5 short ones) |
| `MAX_LEN` | 320 | tokens per input; if anything is cut, it is the start of the history, never the candidate |
| `SCORE_LOSS` | `ordinal` | score questions: `ordinal` (order-aware, Signpost) or `plain` cross-entropy |
| `SEED` | 42 | the first seed; the collapse guard moves on by +1000 per retry |
| `SAVE_MODEL` | 1 | merge LoRA, save `signpost/` + `signpost.zip` in the notebook's working folder, reload and check |
| `LORA_SMOKE` | 0 | 1 = a tiny code check on ModernBERT-base |

The other experiments of runs 19–24 (Nemotron-1B, two specialists, plain vs ordinal) are in Cell 9, and each is reproducible from its own commit in the ledger.

In [1]:
# Cell 1 — Setup: packages, the recipe's settings, device, progress log

!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, os, random, shutil, statistics, subprocess, sys, time
from collections import Counter
from pathlib import Path

import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers

SMOKE = os.environ.get("LORA_SMOKE") == "1"
ENCODER = "answerdotai/ModernBERT-base" if SMOKE else "answerdotai/ModernBERT-large"
LORA_R, LORA_ALPHA, LORA_DROPOUT = int(os.environ.get("LORA_R", "16")), float(os.environ.get("LORA_ALPHA", "32")), 0.1
LR, HEAD_LR = float(os.environ.get("LR", "2e-4")), float(os.environ.get("HEAD_LR", "1e-3"))
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "2"))
BATCH_ROUTE, BATCH_CHOICE = int(os.environ.get("BATCH_ROUTE", "4")), int(os.environ.get("BATCH_CHOICE", "16"))
MAX_LEN = int(os.environ.get("MAX_LEN", "320"))
SCORE_LOSS = os.environ.get("SCORE_LOSS", "ordinal")
SCORE_EPS, SCORE_DIST_W = 0.1, 0.5                       # ordinal: probability for the neighbouring levels; distance weight
SEED = int(os.environ.get("SEED", "42"))
SAVE_MODEL = os.environ.get("SAVE_MODEL", "1") == "1"
SEP, ROUTE_Q = " [SEP] ", "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("lora_signpost_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | {DEVICE}"
    + (f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else "") + f" | {ENCODER}" + (" | SMOKE CHECK" if SMOKE else ""))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 102.8 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 35.1 MB/s eta 0:00:00
[10:32:35] torch 2.11.0+cu128 | transformers 5.17.0 | peft 0.21.2 | cuda:0 (Tesla T4) | answerdotai/ModernBERT-large


---
# Data

## Cell 2 — One question format for routing and questions

Every item becomes `{task, messages, current, question, options {name: description}, answer}`:
- **routing:** the training conversations (templates + realistic) with their domain's 5 agents and the agent holding the chat;
- **questions:** grounded yes / no, choice and score items. The text is one user message, there is no current agent, and options have no description; for score questions the options are the levels, low to high.

**Dev sets** (best epoch and the collapse guard only): 10% of the routing **domains** (unseen companies, as in the tests) and 10% of the question **texts**. The training data was built with guards against near-copies of E12; the independent sets are a final check only, and nothing in training was changed because of them.

**Expect.** The training and dev sizes per task, then the size of every test.

In [2]:
# Cell 2a — Training items in one format, and the dev splits

DATA_BRANCH = "signpost"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():                                   # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

domains = {**load("generic/domains.json"), **load("realistic_train/domains.json")}
route_items = [{"task": "routing", "domain": ex["domain"], "messages": ex["messages"], "current": ex["current_agent"],
                "question": ROUTE_Q, "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]}
               for ex in load("generic/conversations.json") + load("realistic_train/conversations.json")]
choice_items = [{"task": it["type"], "domain": it["domain"], "messages": [{"role": "user", "content": it["state"]}], "current": None,
                 "question": it["question"], "options": {o: "" for o in it["options"]}, "answer": it["answer"], "state": it["state"],
                 "levels": list(it["options"])}            # score questions: the levels, low to high
                for it in load("choice_train/items.json")["items"]]

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items}); rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items}); rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route, dev_route = [x for x in route_items if x["domain"] not in dev_domains], [x for x in route_items if x["domain"] in dev_domains]
train_choice, dev_choice = [x for x in choice_items if x["state"] not in dev_texts], [x for x in choice_items if x["state"] in dev_texts]
if SMOKE:
    train_route, dev_route, train_choice, dev_choice = train_route[::400], dev_route[::100], train_choice[::400], dev_choice[::100]

for name, xs in [("train routing", train_route), ("train questions", train_choice), ("dev routing", dev_route), ("dev questions", dev_choice)]:
    print(f"{name:15s}: {len(xs):5d} | {dict(Counter(x['task'] for x in xs))}")
print(f"dev routing domains (unseen in training): {sorted(dev_domains)}")

Cloning into 'llms-the-hard-way'...


train routing  :  6319 | {'routing': 6319}
train questions:  9586 | {'noul': 5070, 'choice': 2596, 'score': 1920}
dev routing    :   531 | {'routing': 531}
dev questions  :  1014 | {'noul': 550, 'choice': 236, 'score': 228}
dev routing domains (unseen in training): ['campsite', 'hearing_clinic', 'home_cinema_installer', 'hr', 'motorbike_rental', 'museum', 'opticians']


In [3]:
# Cell 2b — The tests

real_convs = load("realistic/conversations.json")                                      # E10
ft = load("first_turn/messages.json")
e11 = [{**m, "agents": ft["companies"][m["company"]]} for m in ft["messages"]]          # E11, with each message's agents
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
ind_q, ind_r, ind_f = load("independent/questions.json")["items"], load("independent/routing.json"), load("independent/first_turn.json")["messages"]
ind2_q, ind2_r, ind2_f = (load("independent2/questions.json")["items"], load("independent2/routing.json"),
                          load("independent2/first_turn.json")["messages"])
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 9
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])         # the 6 test companies of 02, Cell 16
if SMOKE:
    real_convs, e11, banking, e12 = real_convs[:2], e11[::20], banking[:6], e12[::20]
    ind_q, ind_r, ind_f, ind2_q, ind2_r, ind2_f = ind_q[::30], ind_r[:2], ind_f[::20], ind2_q[::20], ind2_r[:2], ind2_f[::10]

turns = lambda convs: sum(len(c["user_turns"]) for c in convs)
print(f"E10 {len(real_convs)} conversations / {turns(real_convs)} turns | E11 {len(e11)} | banking {len(banking)} | E12 {len(e12)}")
print(f"independent: questions {len(ind_q)} | routing {len(ind_r)} conversations / {turns(ind_r)} turns | first messages {len(ind_f)}")
print(f"independent 2: questions {len(ind2_q)} (up to {max(len(q['options']) for q in ind2_q)} options) | routing {len(ind2_r)} "
      f"conversations / {turns(ind2_r)} turns (up to {max(len(c['agents']) for c in ind2_r)} agents) | first messages {len(ind2_f)}")

E10 35 conversations / 193 turns | E11 100 | banking 225 | E12 170
independent: questions 300 | routing 40 conversations / 172 turns | first messages 200
independent 2: questions 200 (up to 50 options) | routing 12 conversations / 70 turns (up to 11 agents) | first messages 50


---
# The model

## Cell 3 — ModernBERT-large + LoRA + a scoring head

**LoRA** keeps every pretrained weight `W` frozen and learns a low-rank update: `y = W x + (α / r) · B (A x)`, with `A` `[r, in]`, `B` `[out, r]`, and `B` starting at **zero**, so training starts from the unchanged encoder. Here it is the `peft` library on `Wqkv`, `Wo` and `Wi` (attention and MLP, all 28 layers: 112 layers, 7.2 M parameters); the mechanism itself is built by hand in the LoRA notebook. At saving, `merge_and_unload()` folds `B·A` into `W`, so Signpost is plain ModernBERT weights.

**Head:** the mean of the output token vectors → `Dropout → Linear(H, 256) → GELU → Dropout → Linear(256, 1)`: one score per option. A question's options go through the encoder in one batch, and a question can have any number of options.

In [4]:
# Cell 3 — The model: encoder with LoRA (peft), scoring head, input text, scores for any number of options

from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer


class Model:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(ENCODER)
        self.tok.padding_side, self.tok.truncation_side = "right", "left"       # if too long, the oldest history is cut
        enc = AutoModel.from_pretrained(ENCODER, dtype=torch.float32).to(DEVICE)
        self.H = enc.config.hidden_size
        self.enc = get_peft_model(enc, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                  target_modules=["Wqkv", "Wo", "Wi"]))
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    @staticmethod
    def text(item, name):
        """messages SEP current agent SEP question SEP candidate: the format signpost.py uses."""
        desc = item["options"][name]
        parts = [f"{m['role']}: {m['content']}" for m in item["messages"]]
        parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}",
                  f"candidate: {name}. {desc}" if desc else f"candidate: {name}"]
        return SEP.join(parts)

    def scores(self, items):
        """-> [items, most options] scores (missing options -1e4): one encoder pass over every option of every item."""
        sizes = torch.tensor([len(it["options"]) for it in items], device=DEVICE)
        texts = [self.text(it, name) for it in items for name in it["options"]]
        enc = self.tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        s = self.head((hidden.float() * m).sum(1) / m.sum(1)).squeeze(-1)           # mean pooling -> one score per option
        real = torch.arange(int(sizes.max()), device=DEVICE) < sizes[:, None]     # [items, K]: which slots hold an option
        out = torch.full(real.shape, -1e4, device=DEVICE)
        out[real] = s                                                             # options in order, item by item
        return out

    def probs(self, item):
        """Probabilities over one item's options, in order; no gradient."""
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

---
# Training

## Cell 4 — Loss, collapse guard, training loop

- **Loss:** listwise cross-entropy over each question's options. Score questions use the **ordinal** variant: 10% of the target goes to the neighbouring levels (a near miss costs less), plus 0.5 × |expected level − true level| / (levels − 1) (a far miss costs more).
- **Training:** each epoch shuffles routing batches and question batches together, and shuffles every question's options, so position can't be learned. AdamW (adapters and head at their own learning rates), 6% warm-up then linear decay, gradient clipping at 1.0, fp16 mixed precision.
- **Best epoch:** after each epoch, both dev sets are scored; the epoch with the best mean is kept.
- **Collapse guard** (run 22): if after epoch 1 dev routing is below 95% or dev questions below 85%, training restarts with the next seed (up to 3 attempts).

In [5]:
# Cell 4a — The ordinal loss (as tensors), dev accuracy, and the collapse guard

def ordinal_loss(scores, items, target):
    """Listwise loss; score questions get SCORE_EPS of the target spread over the neighbouring levels, plus
    SCORE_DIST_W x |expected level - true level| / (levels - 1). Other questions keep the one-hot target."""
    logp = F.log_softmax(scores, dim=1)
    K = scores.shape[1]
    is_score = torch.tensor([it["task"] == "score" for it in items], device=scores.device)
    level = torch.tensor([[it["levels"].index(n) for n in it["options"]] + [-9] * (K - len(it["options"])) if it["task"] == "score"
                          else [-9] * K for it in items], device=scores.device, dtype=scores.dtype)   # each slot's scale position
    true = level.gather(1, target[:, None])                                                      # [B, 1]
    nbr = ((level - true).abs() == 1) & is_score[:, None]                                        # neighbouring levels
    n_nbr = nbr.sum(1, keepdim=True)
    eps = SCORE_EPS * (n_nbr > 0)
    soft = torch.where(nbr, eps / n_nbr.clamp(min=1), F.one_hot(target, K) * (1 - eps))
    loss = -(soft * logp).sum(1).mean()
    if not is_score.any():
        return loss
    n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
    expected = (logp.exp() * level.clamp(min=0)).sum(1)
    return loss + SCORE_DIST_W * ((expected - true[:, 0]).abs() / (n_levels - 1).clamp(min=1))[is_score].mean()


def dev_accuracy(model, items, batch=16):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, model.scores(chunk).argmax(1).tolist()))
    return ok / max(1, len(items)), ok


class Collapsed(Exception):
    """Training collapsed (run 22: the question loss stayed at chance after the warm-up peak); retry with another seed."""


COLLAPSE_DEV = {"routing": 0.95, "questions": 0.85}         # healthy runs had >= 0.977 / 0.949 after epoch 1

In [6]:
# Cell 4b — The training loop

def train_model(model, seed):
    batches = [train_route[i:i + BATCH_ROUTE] for i in range(0, len(train_route), BATCH_ROUTE)]
    batches += [train_choice[i:i + BATCH_CHOICE] for i in range(0, len(train_choice), BATCH_CHOICE)]
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * len(batches)
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0 = (-1.0, None, 0), 0, time.perf_counter()
    log(f"training: {len(batches)} batches per epoch ({len(train_route)} routing + {len(train_choice)} questions), {total} steps, "
        f"score loss {SCORE_LOSS}")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(seed + epoch).shuffle(order)
        run = {"routing": [], "questions": []}
        for bi in order:
            items = [dict(it) for it in batches[bi]]
            for it in items:                                    # options shuffled, so position can't be learned
                names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
            out = model.scores(items)
            loss = ordinal_loss(out, items, target) if SCORE_LOSS == "ordinal" else F.cross_entropy(out, target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run["routing" if items[0]["task"] == "routing" else "questions"].append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  epoch {epoch} step {step}/{total} | loss routing {statistics.mean(run['routing'][-50:] or [0]):.4f} "
                    f"questions {statistics.mean(run['questions'][-50:] or [0]):.4f} | {step / el:.2f} steps/s | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
        (acc_r, ok_r), (acc_c, ok_c) = dev_accuracy(model, dev_route), dev_accuracy(model, dev_choice)
        msg = f"dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%}) | dev questions {ok_c}/{len(dev_choice)} ({acc_c:.1%})"
        log(f"epoch {epoch} done | {msg}")
        if epoch == 1 and (acc_r < COLLAPSE_DEV["routing"] or acc_c < COLLAPSE_DEV["questions"]):
            raise Collapsed(f"{msg} after epoch 1 (thresholds {COLLAPSE_DEV})")
        if (acc_r + acc_c) / 2 > best[0]:
            best = ((acc_r + acc_c) / 2, {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                          "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best[2]} (mean dev accuracy {best[0]:.1%})")
    return {"train_minutes": minutes, "best_epoch": best[2]}

---
# Tests

## Cell 5 — One function per kind of test

Every test question is scored on its own (all its options in one pass) and timed with the GPU synchronized, tokenization included. Four kinds of test, each one function:
- **live routing** (E10, both independent sets): every user turn with the history so far. *Live*: the model's own previous pick is the current agent, so a mistake can carry over. *Given*: the correct previous agent is, so one wrong label can't cascade.
- **first messages** (E11, both independent sets): one message, no history, no current agent; top-1 and top-2.
- **banking**: each conversation with its own current agent.
- **questions** (E12, both independent sets): the text as the user's message; yes / no, choice, score and many-option.

`R` collects every result as (right, total) under one name, so the final table can line it up against run 25 and the other systems.

In [7]:
# Cell 5a — Timing, a table helper, and the four test functions

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def table(rows, key, cols, groups=None):
    """Print right / total per group (key(row) -> group) and for ALL rows; cols = {column: row -> True if right}."""
    groups = list(groups or sorted({key(r) for r in rows})) + ["ALL"]
    w = max(len(str(g_)) for g_ in groups)
    print(f"  {'':{w}s} | {'n':>4} | " + " | ".join(f"{c:>12s}" for c in cols))
    for g_ in groups:
        rs = [r for r in rows if g_ == "ALL" or key(r) == g_]
        if rs:
            print(f"  {g_:{w}s} | {len(rs):>4} | " + " | ".join(f"{sum(map(f_, rs)):>5} / {len(rs):<4}" for f_ in cols.values()))


def user(text):
    return [{"role": "user", "content": text}]


def live_routing(model, convs, given=False):
    rows, lat = [], []
    for conv in convs:
        cur = None
        for u in conv["user_turns"]:
            p, ms = timed_probs(model, {"messages": conv["messages"][:u["message_index"] + 1], "current": cur, "question": ROUTE_Q,
                                        "options": conv["agents"]})
            pick = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv.get("company"), "type": u["turn_type"], "flag": u.get("flag"), "ok": pick == u["expected_agent"]})
            lat.append(ms)
            cur = u["expected_agent"] if given else pick
    return rows, lat


def first_messages(model, msgs):
    rows, lat = [], []
    for m in msgs:
        p, ms = timed_probs(model, {"messages": user(m["message"]), "current": None, "question": ROUTE_Q, "options": m["agents"]})
        names, top = list(m["agents"]), p.topk(2).indices.tolist()
        rows.append({"type": m["type"], "ok": names[top[0]] == m["target_agent"], "top2": m["target_agent"] in [names[i] for i in top]})
        lat.append(ms)
    return rows, lat


def banking_routing(model):
    rows, lat = [], []
    for ex in banking:
        p, ms = timed_probs(model, {"messages": ex["messages"], "current": ex["current_agent"], "question": ROUTE_Q, "options": BANKING_AGENTS})
        rows.append({"ok": list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"]})
        lat.append(ms)
    return rows, lat


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": user(it["state"]), "current": None, "question": it["question"],
                                    "options": {o: "" for o in it["options"]}})
        top = int(p.argmax())
        rows.append({**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"], "conf": p.max().item(),
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [8]:
# Cell 5b — Run every test: R = {result: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}
right = {"right": lambda r: r["ok"]}


def n_ok(rows, key="ok"):
    return sum(r[key] for r in rows), len(rows)


def evaluate(model):
    model.train(False)
    timed_probs(model, {"messages": user("warm-up"), "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    R, LAT = {}, {}
    for name, convs in [("E10", real_convs), ("IND", ind_r), ("IND2", ind2_r)]:
        log(f"{name}: routing every turn")
        rows, LAT[f"{name} routing"] = live_routing(model, convs)
        R[f"{name} routing live"], R[f"{name} routing given"] = n_ok(rows), n_ok(live_routing(model, convs, given=True)[0])
        if name == "E10":
            R["E10 test companies"] = n_ok([r for r in rows if r["company"] in TEST_COMPANIES])
        print(f"{name} routing live, per turn type (given previous agent: {R[f'{name} routing given'][0]})")
        table(rows, lambda r: r["type"], right)
    for name, msgs in [("E11", e11), ("IND", ind_f), ("IND2", ind2_f)]:
        log(f"{name}: first messages")
        rows, LAT[f"{name} first"] = first_messages(model, msgs)
        R[f"{name} first top-1"], R[f"{name} first top-2"] = n_ok(rows), n_ok(rows, "top2")
        table(rows, lambda r: r["type"], {"top-1": lambda r: r["ok"], "top-2": lambda r: r["top2"]})
    log("banking")
    rows, LAT["banking"] = banking_routing(model)
    R["banking 50"], R["banking 225"] = n_ok(rows[:50]), n_ok(rows)
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        log(f"{name}: questions")
        rows, LAT[f"{name} questions"] = questions(model, items)
        R[f"{name} questions"] = n_ok(rows)
        R.update({f"{name} {TYPE[t]}": n_ok([r for r in rows if r["type"] == t]) for t in TYPE if any(r["type"] == t for r in rows)})
        table(rows, lambda r: TYPE[r["type"]], right, [TYPE[t] for t in TYPE])
        sc = [r for r in rows if r["type"] == "score"]
        print(f"  score within 1 level: {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)}")
        if name == "E12":
            table(rows, lambda r: r["category"], right)
            print("  mistakes:")
            for r in [r for r in rows if not r["ok"]]:
                print(f"    {r['id']} {r['type']:6s} | {r['state'][:44]:44s} | {r['question'][:36]:36s} | "
                      f"expected {r['answer']:14s} | got {r['pick']} ({r['conf']:.2f})")
        if name == "IND":
            R["IND questions unflagged"] = n_ok([r for r in rows if not r.get("flag")])
            print(f"  without flagged items {R['IND questions unflagged'][0]} / {R['IND questions unflagged'][1]} | claim-check yes/no "
                  f"(truth always yes) {n_ok([r for r in rows if r.get('flag') == 'claim_check'])[0]}")
    for name, L in LAT.items():
        L = lat_summary(L)
        print(f"latency {name:16s} p50 {L['p50']:4.0f} ms | p95 {L['p95']:4.0f} ms ({L['n']} questions)")
    return R, LAT

---
# Signpost

## Cell 6 — Train and test

The recipe: ModernBERT-large, LoRA r 16, ordinal score loss, 2 epochs, seed 42, with the collapse guard. Then every test.

**Expect** about 30 minutes of training on a T4 and run 25's results within the usual noise (±4 on most tests): E10 78 / 83 and 181 / 193, E11 97, banking 48 · 210, E12 161, independent questions 221, routing 133, first messages 180.

In [9]:
# Cell 6 — Train (a collapse restarts with the next seed), then run every test

attempts = []
for attempt in range(3):
    seed = SEED + 1000 * attempt
    random.seed(seed); torch.manual_seed(seed)
    if AMP:
        torch.cuda.reset_peak_memory_stats(DEVICE)
    model = Model()
    log(f"attempt {attempt + 1} (seed {seed}) | head {sum(p.numel() for p in model.head.parameters()) / 1e6:.2f} M | {gpu()}")
    model.enc.print_trainable_parameters()
    try:
        trained = train_model(model, seed)
        attempts.append({"seed": seed, "status": "ok"})
        break
    except Collapsed as e:
        log(f"attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
        attempts.append({"seed": seed, "status": f"collapsed: {e}"})
        model = None; gc.collect()                              # free the failed attempt
        torch.cuda.empty_cache() if AMP else None
else:
    raise RuntimeError(f"training collapsed in all attempts: {attempts}")

R, LAT = evaluate(model)
peak_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
json.dump({"results": R, "latency": {k: lat_summary(v) for k, v in LAT.items()}, "attempts": attempts, **trained, "peak_gpu_gb": peak_gb},
          open("lora_signpost_results.json", "w"), indent=1)

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[10:32:54] attempt 1 (seed 42) | head 0.26 M | GPU 1.6 GB (peak 1.6)
trainable params: 7,196,672 || all params: 401,978,368 || trainable%: 1.7903
[10:32:54] training: 2180 batches per epoch (6319 routing + 9586 questions), 4360 steps, score loss ordinal
[10:33:39]   epoch 1 step 100/4360 | loss routing 1.5152 questions 0.9764 | 2.19 steps/s | ~32.4 min left | GPU 1.7 GB (peak 8.8)
[10:34:25]   epoch 1 step 200/4360 | loss routing 1.0467 questions 0.8358 | 2.19 steps/s | ~31.6 min left | GPU 1.7 GB (peak 8.9)
[10:35:18]   epoch 1 step 300/4360 | loss routing 0.5068 questions 0.6827 | 2.08 steps/s | ~32.6 min left | GPU 1.7 GB (peak 9.2)
[10:36:08]   epoch 1 step 400/4360 | loss routing 0.1981 questions 0.8271 | 2.05 steps/s | ~32.2 min left | GPU 1.7 GB (peak 9.6)
[10:36:58]   epoch 1 step 500/4360 | loss routing 0.2830 questions 0.7707 | 2.05 steps/s | ~31.4 min left | GPU 1.7 GB (peak 9.6)
[10:37:49]   epoch 1 step 600/4360 | loss routing 0.1333 questions 0.3349 | 2.03 steps/s | ~30.9

## Cell 7 — Save and verify

With `SAVE_MODEL = 1`, the trained model is written to the notebook's own working folder (on Kaggle, `/kaggle/working`, the Output panel). Nothing is published anywhere; uploading to Hugging Face is a separate, manual step.
1. `adapter/`: the LoRA adapter alone (~15 MB, `peft` format), saved **before** merging. With `head.pt` and the base ModernBERT-large it rebuilds the model.
2. `merge_and_unload()` folds every update into its weight (`W ← W + (α / r)·B·A`); `encoder/` holds the merged ModernBERT in fp16 with its tokenizer.
3. `head.pt`, `signpost_config.json` (input format, settings, this run's results) and `signpost.py` (the standalone loader: `route`, `yes_no`, `choice`, `score`); everything zipped to `signpost.zip`.
4. The folder is **reloaded with `signpost.py`** and E11 and E12 are scored again, in fp16 (GPU) and fp32 (what a CPU computes). They must match the trained model's results.

In [10]:
# Cell 7 — Save signpost/ (+ adapter) and signpost.zip, then reload with signpost.py and score E11 / E12 again

OUT = Path("signpost")
if SAVE_MODEL:
    shutil.rmtree(OUT, ignore_errors=True)
    model.enc.save_pretrained(OUT / "adapter")                        # the LoRA adapter alone, before merging
    merged = model.enc.merge_and_unload()                             # W <- W + (alpha / r) B A in all 112 layers
    merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
    model.tok.save_pretrained(OUT / "encoder")
    torch.save({"state_dict": {k: v.float().cpu() for k, v in model.head.state_dict().items()}, "hidden": model.H}, OUT / "head.pt")
    cfg = {"name": "Signpost", "base_model": ENCODER, "license_of_base": "Apache-2.0", "sep": SEP, "prefix": "", "max_len": MAX_LEN,
           "route_question": ROUTE_Q, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora": f"peft {peft.__version__}",
           "score_loss": SCORE_LOSS, "epochs": EPOCHS, "best_epoch": trained["best_epoch"], "seed_attempts": attempts,
           "trained_on": {"routing": len(train_route), "questions": len(train_choice)}, "data_branch": DATA_BRANCH,
           "results": {k: v[0] for k, v in R.items()}}
    json.dump(cfg, open(OUT / "signpost_config.json", "w"), indent=1)
    shutil.copy(DATA_DIR.parent.parent / "signpost" / "signpost.py", OUT / "signpost.py")
    shutil.make_archive("signpost", "zip", root_dir=".", base_dir=str(OUT))
    size = {d: sum(f.stat().st_size for f in (OUT / d).rglob("*") if f.is_file()) / 1e6 for d in ("encoder", "adapter")}
    log(f"saved {OUT}/ (encoder {size['encoder']:.0f} MB, adapter {size['adapter']:.0f} MB) and signpost.zip "
        f"({Path('signpost.zip').stat().st_size / 1e9:.2f} GB)")

    sys.path.insert(0, str(OUT.resolve()))
    from signpost import Signpost
    for dtype in (torch.float16, torch.float32):                     # fp16 = GPU use; fp32 = what a CPU run computes
        sp = Signpost(OUT, device=DEVICE, dtype=dtype)
        n11 = sum(sp.route(user(m["message"]), m["agents"])[0] == m["target_agent"] for m in e11)
        n12 = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"] for it in e12)
        log(f"reload check ({dtype}): E11 {n11}/{len(e11)} (trained model {R['E11 first top-1'][0]}) | "
            f"E12 {n12}/{len(e12)} (trained model {R['E12 questions'][0]})")
        del sp

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[11:14:44] saved signpost/ (encoder 793 MB, adapter 29 MB) and signpost.zip (0.76 GB)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[11:14:52] reload check (torch.float16): E11 96/100 (trained model 96) | E12 162/170 (trained model 162)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[11:15:13] reload check (torch.float32): E11 96/100 (trained model 96) | E12 162/170 (trained model 162)


---
# Results

## Cell 8 — This run next to the published Signpost, the frozen system and the decider

Every column is scored on the same items. Run 25 is the published Signpost (its independent-2 numbers come from `05_listwise_signpost.ipynb`, branch `exp/listwise-index`, which tested the Hub model); the frozen system is `02` (run 17); the decider comes from `04` (full history; it has no current-agent input, so "given" doesn't apply).

In [11]:
# Cell 8 — Side by side, then latency

REF = {
    "Signpost (run 25)": {"E10 test companies": 78, "E10 routing live": 181, "E11 first top-1": 97, "banking 50": 48, "banking 225": 210,
                          "E12 questions": 161, "E12 yes/no": 58, "E12 choice": 58, "E12 score": 45,
                          "IND questions": 221, "IND yes/no": 75, "IND choice": 103, "IND score": 43, "IND routing live": 133,
                          "IND first top-1": 180, "IND2 questions": 188, "IND2 yes/no": 47, "IND2 choice": 47, "IND2 score": 45,
                          "IND2 many-option": 49, "IND2 routing live": 62, "IND2 routing given": 65, "IND2 first top-1": 47},
    "frozen (02, run 17)": {"E10 test companies": 69, "E10 routing live": 158, "E11 first top-1": 94, "banking 50": 43, "banking 225": 194,
                            "E12 questions": 114, "E12 yes/no": 38, "E12 choice": 44, "E12 score": 32},
    "decider 2B": {"E10 test companies": 62, "E10 routing live": 151, "E11 first top-1": 96, "banking 50": 48, "banking 225": 214,
                   "E12 questions": 160, "E12 yes/no": 56, "E12 choice": 56, "E12 score": 48, "IND questions": 253, "IND yes/no": 77,
                   "IND choice": 106, "IND score": 70, "IND questions unflagged": 253, "IND routing live": 113, "IND first top-1": 178,
                   "IND2 questions": 185, "IND2 yes/no": 49, "IND2 choice": 46, "IND2 score": 43, "IND2 many-option": 47,
                   "IND2 routing live": 55, "IND2 first top-1": 50},
}
print(f"{'result':24s} | {'this run':>11} | " + " | ".join(f"{c:>19s}" for c in REF))
for k, (ok_, n) in R.items():
    print(f"{k:24s} | {f'{ok_} / {n}':>11} | " + " | ".join(f"{REF[c].get(k, '—'):>19}" for c in REF))
print(f"\ntraining {trained['train_minutes']:.1f} min, best epoch {trained['best_epoch']} | attempts {[a['status'] for a in attempts]} | "
      f"peak GPU {peak_gb:.1f} GB")

result                   |    this run |   Signpost (run 25) | frozen (02, run 17) |          decider 2B
E10 routing live         |   178 / 193 |                 181 |                 158 |                 151
E10 routing given        |   182 / 193 |                   — |                   — |                   —
E10 test companies       |     77 / 83 |                  78 |                  69 |                  62
IND routing live         |   131 / 172 |                 133 |                   — |                 113
IND routing given        |   140 / 172 |                   — |                   — |                   —
IND2 routing live        |     59 / 70 |                  62 |                   — |                  55
IND2 routing given       |     62 / 70 |                  65 |                   — |                   —
E11 first top-1          |    96 / 100 |                  97 |                  94 |                  96
E11 first top-2          |   100 / 100 |               

---
## Cell 9 — What we tried and dropped

Every number is correct / total on the same test items; the run numbers point to `experiments/modernbert_router_ledger.md`. One seed per run. Three runs of the final recipe (21, 22b, 25) put the **run-to-run noise at about ±4** on most tests, so smaller differences aren't evidence.

| Run | What we tried | Result | Decision |
|---|---|---|---|
| 18 | LoRA + head, **grounded questions only** (no routing) | E12 **114 → 153** / 170 (decider 160) in 9 minutes of training; score answers never more than one level off | The step change of the project: the plateau was the frozen encoder. Kept, and extended to routing |
| 19 | **One model for both jobs**, ModernBERT-large vs Nemotron-1B | Both beat the frozen system and the decider on every routing test with no detector or gate (E10 test 75 and 79 / 83 vs 62). ModernBERT kept E12 at 153 while also routing; Nemotron scored 141 | One combined model. The encoders differ: Nemotron routes better, ModernBERT reads better (Nemotron misread numbers, e.g. "45 cm wide" → over 2 m at 0.99) |
| 20 | **More score training data** (709 → 2,004 items) | E12 score stayed at 43 / 53; urgency got *worse* (8 vs 10 / 13) | Dropped as a lever. A new deadline-urgency family taught a shortcut, "no deadline word → low urgency", and E12's urgency comes from severity ("payouts failing") |
| 21 | **Clean data** (training texts that were near-paraphrases of E12 items reworded, and a guard added) + severity-based urgency, then **plain vs ordinal** score loss | Plain: E12 155, score 46. Ordinal: **E12 161** (decider 160), score 47, E10 test 81 / 83 | Ordinal loss adopted (never worse on score, keeps answers within one level). The guard stays. Routing also moved by 4 between the two variants although the loss only touches score questions: the first measure of fine-tuning noise |
| 22 | The same recipe, rerun | **Training collapsed**: the choice loss stayed at chance after the learning-rate peak; E12 88, dev choice 599 / 1,014 | A **collapse guard**: after epoch 1, if dev routing < 95% or dev choice < 85%, restart with the next seed (up to 3 attempts). The rerun (22b) was healthy and reproduced run 21 within noise (E10 test 80, E12 161) |
| 23 | **Nemotron-1B** on the final recipe, 3 epochs | Routing tied ModernBERT (independent live routing 134 vs 131); questions clearly worse (independent 200 vs 231, score 34 vs 53 / 92); 3× the parameters and training time | ModernBERT-large stays the final encoder |
| 24 | **Two specialists** (routing only, questions only) vs one combined model | No specialist beat the combined model beyond noise; the two biggest gaps favoured the combined one (banking 210 vs 218 / 225, E12 155 vs 161) | **One model.** Joint training doesn't pollute either job and may help both |

**Also dropped once the encoder was fine-tuned:** the context-change detector and the confident-stay gate from `02`. The fine-tuned router handles stays and switches on its own (E10 real switches 20 / 20 in run 21).

**What the independent test showed** (written outside the project, run 22b and 25):
- **Routing holds up:** live routing 133 vs 113 / 172 for the decider (121 with a 3-turn window), mostly on switches.
- **Score precision does not:** independent score 43–53 / 92 across seeds, against the decider's 70. E12's score parity came from test items written in the same style as our training data.
- **Our model switches too readily** on stay-type turns written by someone else (37 vs 48 / 55 for the decider in run 22b).

By the independence rule, none of this was fed back into training. These are the published model's known limits, and they're stated on its model card.

**Since run 25:** LoRA comes from the `peft` library instead of the hand-written `LoRALinear` (the same layers, initialization, scale and dropout; the mechanism is built by hand in the LoRA notebook), and the notebook tests on the second independent set too.

---
## Next

The published Signpost (run 25) is on the Hugging Face Hub as **[skamalj/signpost](https://huggingface.co/skamalj/signpost)**, with the standalone loader `signpost.py` (`route`, `yes_no`, `choice`, `score`). The model card's examples were run on the published files. `04_decider_benchmark.ipynb` produces the decider's side of every comparison above.

**Part 5** (branch `signpost-apps`) uses Signpost as the router of a real multi-agent app: LangGraph with five Claude agents, a DynamoDB checkpointer and a message reducer.